# 07 - LLM 内部指标 (LLM Internals)

**目的**: 精确测量 LLM 调用的每个内部阶段，找出真实的延迟构成  
**测量对象**: APIPro (Claude/Qwen/DeepSeek) + Ollama (本地)

**延迟分解**:
```
LLM 调用总延迟
  = [A] 网络连接建立 (TCP handshake)
  + [B] 请求发送 (request payload upload)
  + [C] TTFB：Time To First Byte (provider 处理时间)
  + [D] 生成阶段 (streaming token generation)
  + [E] 最后字节接收 (TTLB)
```

**关键指标**:
- **TTFB**: 首字节时间 = 服务端真实处理延迟（排除网络）
- **tokens/sec**: 生成速率 = 实际推理效率
- **Failover 时间**: Provider 切换耗时（影响用户感知）

## 0. 配置参数

In [ ]:
import os
from datetime import datetime

# =============================================
# Provider 配置
# =============================================
APIPRO_BASE_URL = os.getenv("APIPRO_BASE_URL", "https://api.apipro.ai/v1")
APIPRO_API_KEY  = os.getenv("APIPRO_API_KEY", "")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# 测试哪些 Provider
PROVIDERS = [
    {
        "id": "apipro_claude",
        "label": "APIPro/Claude Sonnet",
        "type": "apipro",
        "model": "claude-sonnet-4-5-20250929",
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "apipro_qwen",
        "label": "APIPro/Qwen Plus",
        "type": "apipro",
        "model": "qwen-plus",
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "apipro_deepseek",
        "label": "APIPro/DeepSeek Chat",
        "type": "apipro",
        "model": "deepseek-chat",
        "enabled": bool(APIPRO_API_KEY),
    },
    {
        "id": "ollama_14b",
        "label": "Ollama/qwen2.5:14b",
        "type": "ollama",
        "model": "qwen2.5:14b",
        "enabled": True,
    },
    {
        "id": "ollama_7b",
        "label": "Ollama/qwen2.5:7b",
        "type": "ollama",
        "model": "qwen2.5:7b",
        "enabled": True,
    },
]

# 测试 Prompt（固定内容，方便对比）
TEST_PROMPT_SHORT = "用一句话描述你自己。"
TEST_PROMPT_MEDIUM = "我需要采购100台服务器，请问需要考虑哪些关键参数？请用列表格式列出主要考量因素。"
ACTIVE_PROMPT = TEST_PROMPT_MEDIUM  # 切换此处选择测试 Prompt
MAX_TOKENS = 200  # 控制输出长度，确保可比性

# 测试轮数（建议 5-10，LLM 调用耗时较长）
N_ITERATIONS = 5

# 是否测试流式输出
TEST_STREAMING = True

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = f"{DATA_DIR}/llm_internals_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

enabled_providers = [p for p in PROVIDERS if p["enabled"]]
print(f"已启用 Provider: {[p['label'] for p in enabled_providers]}")
print(f"测试 Prompt: {ACTIVE_PROMPT[:50]}...")
print(f"Max tokens: {MAX_TOKENS}")

## 1. 依赖导入

In [ ]:
import httpx
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. 非流式调用：总延迟 + Token 用量

In [ ]:
def call_apipro_non_stream(model: str, prompt: str, max_tokens: int, api_key: str, base_url: str) -> dict:
    """APIPro 非流式调用，精确测量 TTFB + 总延迟"""
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "stream": False}

    t_start = time.perf_counter()
    try:
        with httpx.Client(timeout=90.0) as client:
            resp = client.post(f"{base_url}/chat/completions", json=payload, headers=headers)
        t_total = (time.perf_counter() - t_start) * 1000

        if resp.status_code == 200:
            data = resp.json()
            usage = data.get("usage", {})
            content = data["choices"][0]["message"]["content"]
            output_tokens = usage.get("completion_tokens", 0)
            tokens_per_sec = (output_tokens / t_total * 1000) if t_total > 0 else 0

            return {"success": True, "latency_ms": t_total, "ttfb_ms": None,  # 非流式无法测 TTFB
                    "input_tokens": usage.get("prompt_tokens", 0),
                    "output_tokens": output_tokens, "tokens_per_sec": round(tokens_per_sec, 1),
                    "output_preview": content[:80]}
        return {"success": False, "latency_ms": t_total, "error": f"HTTP {resp.status_code}: {resp.text[:200]}"}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - t_start)*1000, "error": str(e)}


def call_apipro_streaming(model: str, prompt: str, max_tokens: int, api_key: str, base_url: str) -> dict:
    """APIPro 流式调用，精确测量 TTFB 和生成速率"""
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json",
               "Accept": "text/event-stream"}
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}],
               "max_tokens": max_tokens, "stream": True}

    t_start = time.perf_counter()
    t_first_token = None
    output_tokens = 0
    content_parts = []

    try:
        with httpx.Client(timeout=90.0) as client:
            with client.stream("POST", f"{base_url}/chat/completions",
                               json=payload, headers=headers) as response:
                if response.status_code != 200:
                    t_total = (time.perf_counter() - t_start) * 1000
                    return {"success": False, "latency_ms": t_total, "error": f"HTTP {response.status_code}"}

                for line in response.iter_lines():
                    if not line or not line.startswith("data: "):
                        continue
                    data_str = line[6:]
                    if data_str == "[DONE]":
                        break
                    try:
                        chunk = json.loads(data_str)
                        delta = chunk["choices"][0].get("delta", {})
                        token_text = delta.get("content", "")
                        if token_text:
                            if t_first_token is None:
                                t_first_token = time.perf_counter()
                            output_tokens += 1
                            content_parts.append(token_text)
                    except Exception:
                        continue

        t_total = (time.perf_counter() - t_start) * 1000
        ttfb = (t_first_token - t_start) * 1000 if t_first_token else t_total
        generation_time = t_total - ttfb
        tokens_per_sec = (output_tokens / generation_time * 1000) if generation_time > 0 else 0

        return {
            "success": True, "latency_ms": t_total, "ttfb_ms": round(ttfb, 1),
            "generation_ms": round(generation_time, 1),
            "output_tokens": output_tokens, "tokens_per_sec": round(tokens_per_sec, 1),
            "output_preview": "".join(content_parts)[:80],
        }
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - t_start)*1000, "error": str(e)}


def call_ollama_streaming(model: str, prompt: str, base_url: str) -> dict:
    """Ollama 流式调用，测量 TTFB 和生成速率"""
    payload = {"model": model, "messages": [{"role": "user", "content": prompt}], "stream": True}

    t_start = time.perf_counter()
    t_first_token = None
    output_tokens = 0
    content_parts = []

    try:
        with httpx.Client(timeout=120.0) as client:
            with client.stream("POST", f"{base_url}/api/chat", json=payload) as response:
                if response.status_code != 200:
                    return {"success": False, "latency_ms": (time.perf_counter() - t_start)*1000,
                            "error": f"HTTP {response.status_code}: {response.text[:100]}"}

                for line in response.iter_lines():
                    if not line:
                        continue
                    try:
                        chunk = json.loads(line)
                        content = chunk.get("message", {}).get("content", "")
                        if content:
                            if t_first_token is None:
                                t_first_token = time.perf_counter()
                            output_tokens += 1
                            content_parts.append(content)
                        if chunk.get("done"):
                            # Ollama 在 done=True 时提供精确的 token 统计
                            output_tokens = chunk.get("eval_count", output_tokens)
                            break
                    except Exception:
                        continue

        t_total = (time.perf_counter() - t_start) * 1000
        ttfb = (t_first_token - t_start) * 1000 if t_first_token else t_total
        generation_time = t_total - ttfb
        tokens_per_sec = (output_tokens / generation_time * 1000) if generation_time > 0 else 0

        return {
            "success": True, "latency_ms": t_total, "ttfb_ms": round(ttfb, 1),
            "generation_ms": round(generation_time, 1),
            "output_tokens": output_tokens, "tokens_per_sec": round(tokens_per_sec, 1),
            "output_preview": "".join(content_parts)[:80],
        }
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - t_start)*1000, "error": str(e)}


print("LLM 调用函数加载完成")

## 3. 非流式 vs 流式对比

In [ ]:
all_results = []

print(f"LLM 内部指标测试 ({N_ITERATIONS} 轮)")
print("=" * 75)

for prov in enabled_providers:
    print(f"\n[{prov['label']}]")

    for i in range(N_ITERATIONS):
        # 流式调用（测量 TTFB）
        if TEST_STREAMING:
            if prov["type"] == "apipro":
                r_stream = call_apipro_streaming(prov["model"], ACTIVE_PROMPT, MAX_TOKENS, APIPRO_API_KEY, APIPRO_BASE_URL)
            else:
                r_stream = call_ollama_streaming(prov["model"], ACTIVE_PROMPT, OLLAMA_BASE_URL)

            record_stream = {
                "provider_id": prov["id"],
                "provider_label": prov["label"],
                "mode": "streaming",
                "iteration": i + 1,
                "latency_ms": r_stream.get("latency_ms", 0),
                "ttfb_ms": r_stream.get("ttfb_ms"),
                "generation_ms": r_stream.get("generation_ms"),
                "output_tokens": r_stream.get("output_tokens", 0),
                "tokens_per_sec": r_stream.get("tokens_per_sec", 0),
                "success": r_stream.get("success", False),
            }
            all_results.append(record_stream)

            status = "OK" if r_stream["success"] else "FAIL"
            ttfb_str = f"TTFB:{r_stream.get('ttfb_ms', 'N/A'):.0f}ms" if r_stream.get("ttfb_ms") else "TTFB:N/A"
            print(f"  [{status}] 轮{i+1:02d} stream | 总延迟:{r_stream['latency_ms']:.0f}ms | {ttfb_str} | {r_stream.get('tokens_per_sec', 0):.1f} tok/s | {r_stream.get('output_tokens', 0)} tokens")

            if not r_stream["success"] and r_stream.get("error"):
                print(f"         错误: {r_stream['error'][:100]}")

df_results = pd.DataFrame(all_results)
print("\n测试完成")

## 4. 统计分析

In [ ]:
df_ok = df_results[df_results["success"]]

if df_ok.empty:
    print("无成功记录，请检查 Provider 配置")
else:
    print("=" * 75)
    print("LLM 内部指标汇总")
    print("=" * 75)

    summary = df_ok.groupby(["provider_label", "mode"]).agg(
        total_p50=("latency_ms", lambda x: np.percentile(x, 50)),
        total_p95=("latency_ms", lambda x: np.percentile(x, 95)),
        ttfb_p50=("ttfb_ms", lambda x: np.percentile(x.dropna(), 50) if len(x.dropna()) > 0 else None),
        tokens_per_sec_mean=("tokens_per_sec", "mean"),
        output_tokens_mean=("output_tokens", "mean"),
        count=("success", "count"),
    ).round(1)

    print(summary.to_string())

    print("\n=" * 75)
    print("TTFB 分析（排除网络传输的纯 Provider 处理时间）")
    print("=" * 75)
    for label, grp in df_ok.groupby("provider_label"):
        ttfb_data = grp["ttfb_ms"].dropna()
        if len(ttfb_data) > 0:
            print(f"  {label:35s}: TTFB P50={np.percentile(ttfb_data, 50):.0f}ms, P95={np.percentile(ttfb_data, 95):.0f}ms")

    print("\n=" * 75)
    print("生成速率（tokens/sec）")
    print("=" * 75)
    for label, grp in df_ok.groupby("provider_label"):
        tps_data = grp["tokens_per_sec"].dropna()
        if len(tps_data) > 0:
            print(f"  {label:35s}: {tps_data.mean():.1f} tok/s (平均), {tps_data.max():.1f} tok/s (最高)")

## 5. Provider 切换开销（Failover 模拟）

In [ ]:
print("=" * 60)
print("Failover 切换延迟模拟")
print("=" * 60)
print("模拟场景: 首选 Provider 超时(30s)后切换到备选 Provider")
print()

# 当前 manager.py 的 failover 逻辑
# 理论最坏情况 = sum(各 Provider 超时时间)
PROVIDER_TIMEOUTS = {
    "APIPro (primary)": 30,   # 秒
    "APIPro (retry #1)": 30,
    "Ollama (fallback)": 60,
    "Mock (last resort)": 0,
}

print("当前 failover 链（manager.py 配置）:")
total_worst = 0
for i, (provider, timeout_s) in enumerate(PROVIDER_TIMEOUTS.items()):
    if timeout_s > 0:
        total_worst += timeout_s
        print(f"  {i+1}. {provider:30s} timeout={timeout_s}s  (累计最坏={total_worst}s)")
    else:
        print(f"  {i+1}. {provider:30s} 即时可用")

print()
print(f"最坏情况总等待: {total_worst}s ({total_worst//60}分{total_worst%60}秒)")
print()
print("这就是为什么 SESSION 会有 5 分钟等待的根本原因")
print()
print("优化方案 (ENGINE-034):")
print("  - 将 APIPro 超时从 30s → 10s")
print("  - 并发探测多个 Provider，选最快响应")
print("  - 基于历史延迟动态选择 Provider")

## 6. 可视化

In [ ]:
if df_ok.empty:
    print("无数据，跳过可视化")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("LLM 内部指标分析", fontsize=14)

    providers_list = df_ok["provider_label"].unique()

    # 图1: 总延迟 + TTFB 分解柱状图
    ax1 = axes[0]
    total_p50 = [df_ok[df_ok["provider_label"] == p]["latency_ms"].quantile(0.5) for p in providers_list]
    ttfb_p50  = [df_ok[df_ok["provider_label"] == p]["ttfb_ms"].dropna().quantile(0.5) if len(df_ok[df_ok["provider_label"] == p]["ttfb_ms"].dropna()) > 0 else 0 for p in providers_list]
    gen_p50   = [t - ttfb for t, ttfb in zip(total_p50, ttfb_p50)]

    x = range(len(providers_list))
    ax1.bar(x, ttfb_p50, label="TTFB (处理)", color="coral", alpha=0.8)
    ax1.bar(x, gen_p50, bottom=ttfb_p50, label="生成阶段", color="steelblue", alpha=0.8)
    ax1.set_xticks(list(x))
    ax1.set_xticklabels([p.split("/")[-1] for p in providers_list], rotation=20, ha="right", fontsize=8)
    ax1.set_ylabel("延迟 (ms)")
    ax1.set_title("总延迟分解 (P50)")
    ax1.legend()

    # 图2: tokens/sec 对比
    ax2 = axes[1]
    tps_vals = [df_ok[df_ok["provider_label"] == p]["tokens_per_sec"].mean() for p in providers_list]
    bars = ax2.bar(range(len(providers_list)), tps_vals, color="teal", alpha=0.8)
    ax2.set_xticks(range(len(providers_list)))
    ax2.set_xticklabels([p.split("/")[-1] for p in providers_list], rotation=20, ha="right", fontsize=8)
    ax2.set_ylabel("tokens/sec")
    ax2.set_title("生成速率")
    for bar, val in zip(bars, tps_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{val:.1f}",
                 ha='center', va='bottom', fontsize=8)

    # 图3: TTFB 箱线图（用户感知的首响应时间）
    ax3 = axes[2]
    ttfb_data = [df_ok[df_ok["provider_label"] == p]["ttfb_ms"].dropna().values for p in providers_list]
    valid_data = [(d, p) for d, p in zip(ttfb_data, providers_list) if len(d) > 0]
    if valid_data:
        data3, labels3 = zip(*valid_data)
        ax3.boxplot(list(data3), labels=[l.split("/")[-1] for l in labels3], patch_artist=True)
        ax3.set_ylabel("TTFB (ms)")
        ax3.set_title("首字节时间 (TTFB)")
        ax3.tick_params(axis='x', rotation=15)
    else:
        ax3.text(0.5, 0.5, "TTFB 数据不足\n(需要流式调用)", ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title("首字节时间 (TTFB)")

    plt.tight_layout()
    chart_file = f"../data/llm_internals_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
    plt.savefig(chart_file, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"图表已保存: {chart_file}")

## 7. 保存结果

In [ ]:
if not df_results.empty:
    df_results.to_csv(RESULTS_FILE, index=False)
    print(f"结果已保存: {RESULTS_FILE}")
    print(f"共 {len(df_results)} 条记录，{df_results['success'].sum()} 条成功")
else:
    print("无结果保存")